# EDM-Fuzzy / JSD-Fuzzy + ANN-LM — Skema Ladder (sesuai skema pembimbing)

Notebook ini menyelaraskan implementasi dengan *Skema Pengolahan Data Multisource dan Fault Detection*:
- **Broker** = kumpulkan 4 sensor jadi **satu dataset** (identitas sensor tetap; TIDAK difusikan). Tabel gabungan = output broker.
- **4 deret waktu sensor** → windowing → **EDM-Fuzzy per sensor, τ=1..10** → **40 fitur** (4×10) → ANN.
- **5 skenario berdasarkan jumlah fault** (multiplicity): S1 biner, S2 single, S3 two-fault, S4 three-fault, S5 four-fault.
- **Block-split** (leakage-safe): train = 75% waktu awal, test = 25% waktu akhir per kondisi → window train/test tak pernah overlap.
- **ANN-LM**: sklearn tak punya Levenberg-Marquardt; dipakai `solver='lbfgs'` (quasi-Newton, paling dekat ke LM). *Untuk LM asli → MATLAB `trainlm`.*

Metode: **EDM-Fuzzy** (skema) + **JSD-Fuzzy** (usulan paper) supaya sekaligus jadi perbandingan.

In [ ]:
# === Runtime guard ===
import os, time
for _v in ("OMP_NUM_THREADS","OPENBLAS_NUM_THREADS","MKL_NUM_THREADS","NUMEXPR_NUM_THREADS","VECLIB_MAXIMUM_THREADS"):
    os.environ.setdefault(_v,"1")
NOTEBOOK_START=time.time(); KAGGLE_TIME_BUDGET_H=float(os.environ.get("KAGGLE_TIME_BUDGET_H",10.5))
def elapsed_s(): return time.time()-NOTEBOOK_START
def budget_ok(need_s=0.0,label=""):
    left=KAGGLE_TIME_BUDGET_H*3600.0-elapsed_s()
    if left<need_s: print(f"[budget] SKIP {label}: {left/60:.1f} min left"); return False
    return True
def log_stage(x): print(f"[t+{elapsed_s()/60:5.1f} min] {x}",flush=True)
log_stage("guard armed")

In [ ]:
# === Imports + config ===
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings
import requests
from io import StringIO
from numpy.lib.stride_tricks import sliding_window_view
from joblib import Parallel, delayed
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, ConfusionMatrixDisplay)
N_JOBS=-1; RANDOM_SEED=42
DS=4; WIN=512; STRIDE=64; TRAIN_FRAC=0.75      # block-split: 75% waktu awal = train
MAX_PER_CLASS_TR=300; MAX_PER_CLASS_TE=120
S=10; scales=np.arange(1,S+1); m=2; r_ratio=0.2; n_ref=128; jsd_bins=40
METHODS=["EDM-Fuzzy","JSD-Fuzzy"]
# ANN-LM: sklearn tak punya Levenberg-Marquardt -> lbfgs (quasi-Newton) paling dekat.
ANN_SOLVER="lbfgs"; ANN_MAX_ITER=1000
os.makedirs("exports",exist_ok=True)
print("WIN",WIN,"STRIDE",STRIDE,"DS",DS,"| block-split train_frac",TRAIN_FRAC,"| solver",ANN_SOLVER)

In [ ]:
# === Load data (broker output = satu tabel gabungan, 4 kolom) ===
def load_default_data():
    url="https://raw.githubusercontent.com/vousmeevoyez/public-files/refs/heads/main/tabel_sensor4_generated.csv"
    r=requests.get(url); r.raise_for_status(); return pd.read_csv(StringIO(r.text))
df=load_default_data(); cols=["kelembaban1","kelembaban2","kelembaban3","kelembaban4"]
X_df=pd.DataFrame(df[cols].to_numpy(dtype=float),columns=cols).ffill().bfill()
X_df=X_df.fillna(X_df.median(numeric_only=True)); X=X_df.to_numpy(); X_ds=X[::DS]
print("combined table",X.shape,"-> downsampled",X_ds.shape)

In [ ]:
# === Fault simulators + kondisi (identik notebook utama) ===
def simulate_drift_fault(x,intensity=0.02,seed=None):
    d=np.arange(len(x))*intensity; return x+d, np.abs(d)>1e-6
def simulate_spike_fault(x,intensity=0.08,p=0.015,seed=None):
    tau=max(1,int(1.0/p)) if p>0 else len(x)
    sp=(np.arange(len(x))%tau==0).astype(float)*(intensity*np.nanstd(x)); return x+sp, sp!=0
def simulate_bias_fault(x,bias=0.08,seed=None): return x+bias, np.ones(len(x),bool)
def simulate_hardware_fault(x,stuck_prob=0.08,loss_prob=0.05,seed=None):
    rng=np.random.default_rng(seed); n=len(x); rv=rng.random(n); idx=rng.integers(n,size=n)
    m1=rv<stuck_prob; y=x.copy(); y[m1]=x[idx[m1]]; m2=rv<loss_prob; y[m2]=np.nan; return y,(m1|m2)
def simulate_multiple_faults(x,faults,seed=None):
    y=x.copy(); mm=np.zeros(len(x),bool)
    for f,kw in faults: y,mi=f(y,**kw,seed=seed); mm|=mi
    return y,mm
def simulate_choose_one(x,options,seed=None):
    rng=np.random.default_rng(seed); f,kw=options[rng.integers(len(options))]; return f(x,**kw,seed=seed)
SCEN={
 "faulty":[(simulate_choose_one,{"options":[(simulate_drift_fault,{"intensity":0.02}),(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_bias_fault,{"bias":0.08}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})]})],
 "drift":[(simulate_drift_fault,{"intensity":0.02})],"spike":[(simulate_spike_fault,{"intensity":0.08,"p":0.015})],
 "bias":[(simulate_bias_fault,{"bias":0.08})],"hardware":[(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "bias+malfunc":[(simulate_bias_fault,{"bias":0.08}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "spike+malfunc":[(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "spike+bias":[(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_bias_fault,{"bias":0.08})],
 "drift+malfunc":[(simulate_drift_fault,{"intensity":0.02}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "drift+bias":[(simulate_drift_fault,{"intensity":0.02}),(simulate_bias_fault,{"bias":0.08})],
 "drift+spike":[(simulate_drift_fault,{"intensity":0.02}),(simulate_spike_fault,{"intensity":0.08,"p":0.015})],
 "spike+bias+malfunc":[(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_bias_fault,{"bias":0.08}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "drift+bias+malfunc":[(simulate_drift_fault,{"intensity":0.02}),(simulate_bias_fault,{"bias":0.08}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "spike+drift+malfunc":[(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_drift_fault,{"intensity":0.02}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05})],
 "drift+spike+bias":[(simulate_drift_fault,{"intensity":0.02}),(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_bias_fault,{"bias":0.08})],
 "spike+bias+malfunc+drift":[(simulate_spike_fault,{"intensity":0.08,"p":0.015}),(simulate_bias_fault,{"bias":0.08}),(simulate_hardware_fault,{"stuck_prob":0.08,"loss_prob":0.05}),(simulate_drift_fault,{"intensity":0.02})],
}
def inject_multisensor(Xin,faults,seed=0):
    rng=np.random.default_rng(seed); Y=Xin.copy()
    for s in range(Y.shape[1]):
        y,_=simulate_multiple_faults(Y[:,s],faults,seed=int(rng.integers(1e9))); Y[:,s]=y
    Ydf=pd.DataFrame(Y).ffill().bfill(); Ydf=Ydf.fillna(Ydf.median(numeric_only=True)); return Ydf.to_numpy()
print("kondisi:",1+len(SCEN))

In [ ]:
# === Entropy (EDM-Fuzzy + JSD-Fuzzy) + feature builder ===
def make_windows(Xn,win,stride):
    Xn=np.asarray(Xn,dtype=np.float32); N=Xn.shape[0]
    if N<win: return np.empty((0,win,Xn.shape[1]),np.float32)
    v=sliding_window_view(Xn,window_shape=win,axis=0)[np.arange(0,N-win+1,stride,int)]
    return v.transpose(0,2,1)   # (nwin, win, channels)
def coarse_grain_mean(x,s):
    n=(len(x)//s)*s; return x[:n].reshape(-1,s).mean(1) if n>0 else np.array([],float)
def embed(y,mm):
    return np.lib.stride_tricks.sliding_window_view(y,mm) if len(y)>=mm else np.empty((0,mm),float)
def fuzzy_phi(V,r,n_ref=256,seed=0):
    rng=np.random.default_rng(seed); N=V.shape[0]
    if N<3: return np.nan
    ref=rng.choice(N,size=n_ref,replace=False) if N>n_ref else np.arange(N)
    A=V[ref]; d2=np.maximum(np.sum(A*A,1,keepdims=True)+np.sum(V*V,1,keepdims=True).T-2*(A@V.T),0.0)
    mu=1.0/(1.0+d2/(r*r+1e-24)); mu[np.arange(len(ref)),ref]=0.0; return (mu.sum(1)/(N-1)).mean()
def fuzzy_sim(V,r,n_ref=256,seed=0):
    rng=np.random.default_rng(seed); N=V.shape[0]
    if N<3: return np.array([],float)
    ref=rng.choice(np.arange(N),size=n_ref,replace=False) if N>n_ref else np.arange(N)
    A=V[ref]; d=np.sqrt(np.maximum(np.sum(A*A,1,keepdims=True)+np.sum(V*V,1,keepdims=True).T-2*(A@V.T),0.0))
    mu=1.0/(1.0+(d/(r+1e-12))**2)
    for ri,i in enumerate(ref): mu[ri,i]=np.nan
    return mu[~np.isnan(mu)].ravel()
def edm_1d(x,scales,seed=0):
    out=[]
    for s in scales:
        y=coarse_grain_mean(x,s)
        if len(y)<(m+2): out.append(np.nan); continue
        r=r_ratio*np.std(y,ddof=1); pm=fuzzy_phi(embed(y,m),r,n_ref,seed+11*s); pm1=fuzzy_phi(embed(y,m+1),r,n_ref,seed+17*s)
        out.append(np.log(pm/pm1) if (pm and pm1 and pm>0 and pm1>0 and not np.isnan(pm) and not np.isnan(pm1)) else np.nan)
    return np.array(out,float)
def jsd_1d(x,scales,seed=0,bins=20):
    out=[]; be=np.linspace(0,1,bins+1); eps=1e-12
    for s in scales:
        y=coarse_grain_mean(x,s)
        if len(y)<(m+2): out.extend([np.nan]*4); continue
        r=r_ratio*np.std(y,ddof=1); a=fuzzy_sim(embed(y,m),r,n_ref,seed+11*s); b=fuzzy_sim(embed(y,m+1),r,n_ref,seed+17*s)
        if len(a)==0 or len(b)==0: out.extend([np.nan]*4); continue
        p,_=np.histogram(a,bins=be); q,_=np.histogram(b,bins=be); p=p.astype(float); q=q.astype(float)
        if p.sum()==0 or q.sum()==0: out.extend([np.nan]*4); continue
        p/=p.sum(); q/=q.sum(); mid=0.5*(p+q)
        jsd=0.5*(np.sum(p*np.log((p+eps)/(mid+eps)))+np.sum(q*np.log((q+eps)/(mid+eps))))
        out.extend([jsd,np.log((a.mean()+eps)/(b.mean()+eps)),a.mean(),a.std()])
    return np.array(out,float)
def feats(W,method,seed=7):
    if W.shape[0]==0:
        ncol=(4 if method=="JSD-Fuzzy" else 1)*len(scales)*W.shape[2]; return np.empty((0,ncol),float)
    ns=W.shape[2]
    def one(i): return np.concatenate([(jsd_1d(W[i,:,s],scales,seed+19*s,jsd_bins) if method=="JSD-Fuzzy" else edm_1d(W[i,:,s],scales,seed+19*s)) for s in range(ns)])
    F=np.vstack(Parallel(n_jobs=N_JOBS,prefer='processes')(delayed(one)(i) for i in range(W.shape[0])))
    Fdf=pd.DataFrame(F); return Fdf.fillna(Fdf.median(numeric_only=True)).fillna(0.0).to_numpy()
print("entropy fns ready")

In [ ]:
# === Block-split window+feature per kondisi (leakage-safe) ===
# train = 75% waktu awal tiap kondisi, test = 25% akhir -> window tak overlap antar split.
conditions=["normal"]+list(SCEN.keys())
series={"normal":X_ds}
for k,(name,f) in enumerate(SCEN.items(),1): series[name]=inject_multisensor(X_ds,f,seed=100+k)
FT={}   # (method, cond, 'tr'/'te') -> feature matrix
for meth in METHODS:
    for cond in conditions:
        ts=series[cond]; cut=int(TRAIN_FRAC*len(ts))
        Wtr=make_windows(ts[:cut],WIN,STRIDE); Wte=make_windows(ts[cut:],WIN,STRIDE)
        FT[(meth,cond,'tr')]=feats(Wtr,meth); FT[(meth,cond,'te')]=feats(Wte,meth)
    log_stage(f"features done: {meth}")
print("contoh dims EDM normal:",FT[('EDM-Fuzzy','normal','tr')].shape, FT[('EDM-Fuzzy','normal','te')].shape)

In [ ]:
# === Definisi 5 skenario (ladder by fault multiplicity, sesuai PDF) ===
ALL_FAULT=[c for c in conditions if c!="normal"]
LADDER={
 "S1_NormalvsFaulty":[("normal",["normal"]),("faulty",ALL_FAULT)],
 "S2_SingleFault":[("normal",["normal"]),("drift",["drift"]),("spike",["spike"]),("bias",["bias"]),("hardware",["hardware"])],
 "S3_TwoFault":[("normal",["normal"]),("bias+HW",["bias+malfunc"]),("drift+bias",["drift+bias"]),("drift+HW",["drift+malfunc"]),("spike+bias",["spike+bias"]),("drift+spike",["drift+spike"]),("spike+HW",["spike+malfunc"])],
 "S4_ThreeFault":[("normal",["normal"]),("drift+bias+HW",["drift+bias+malfunc"]),("drift+spike+bias",["drift+spike+bias"]),("spike+bias+HW",["spike+bias+malfunc"]),("spike+drift+HW",["spike+drift+malfunc"])],
 "S5_FourFault":[("normal",["normal"]),("drift+spike+bias+HW",["spike+bias+malfunc+drift"])],
}
for sc,cl in LADDER.items(): print(f"{sc}: {len(cl)} kelas -> {[c[0] for c in cl]}")

In [ ]:
# === ANN-LM (lbfgs) grid + evaluasi per skenario per metode (block-split) ===
def cap(X,y,maxp,seed=0):
    rng=np.random.default_rng(seed); keep=[]
    for c in np.unique(y):
        idx=np.where(y==c)[0]
        if len(idx)>maxp: idx=rng.choice(idx,size=maxp,replace=False)
        keep.append(idx)
    keep=np.concatenate(keep); rng.shuffle(keep); return X[keep],y[keep]
# grid arsitektur sesuai PDF (subset alpha/act utk runtime; solver lbfgs)
HIDDEN=[(32,),(64,),(128,),(256,),(64,16),(64,32),(64,64),(128,32),(128,64),(128,128),(256,64),(256,128),(128,64,32)]
def build_scenario(method,classes,split):
    Xs,ys=[],[]
    for ci,(cname,conds) in enumerate(classes):
        for cond in conds:
            F=FT[(method,cond,split)]
            if len(F): Xs.append(F); ys.append(np.full(len(F),ci,int))
    return np.vstack(Xs), np.concatenate(ys)
def run_scenario(method,scname,classes):
    Xtr,ytr=build_scenario(method,classes,'tr'); Xte,yte=build_scenario(method,classes,'te')
    Xtr,ytr=cap(Xtr,ytr,MAX_PER_CLASS_TR,seed=RANDOM_SEED)
    Xte,yte=cap(Xte,yte,MAX_PER_CLASS_TE,seed=RANDOM_SEED+1)
    I=Xtr.shape[1]; O=len(np.unique(ytr))
    pipe=Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler()),
                   ("mlp",MLPClassifier(solver=ANN_SOLVER,max_iter=ANN_MAX_ITER,random_state=RANDOM_SEED))])
    grid={"mlp__hidden_layer_sizes":HIDDEN,"mlp__activation":["relu","tanh"],"mlp__alpha":[1e-3]}
    gs=GridSearchCV(pipe,grid,cv=3,scoring="f1_macro",n_jobs=N_JOBS)
    t0=time.perf_counter(); gs.fit(Xtr,ytr); wall=time.perf_counter()-t0
    best=gs.best_estimator_; pred=best.predict(Xte)
    hl=best.named_steps["mlp"].hidden_layer_sizes
    return dict(Scenario=scname,Method=method,n_classes=O,n_input=I,n_train=len(ytr),n_test=len(yte),
        Best_Hidden=str(hl),Best_Act=best.named_steps["mlp"].activation,
        Accuracy=round(accuracy_score(yte,pred),3),
        Precision=round(precision_score(yte,pred,average="macro",zero_division=0),3),
        Recall=round(recall_score(yte,pred,average="macro",zero_division=0),3),
        F1_macro=round(f1_score(yte,pred,average="macro",zero_division=0),3),
        wall_s=round(wall,1)), (yte,pred,[c[0] for c in classes])

results=[]; cms={}
for scname,classes in LADDER.items():
    for meth in METHODS:
        if not budget_ok(300,f"{scname}/{meth}"): continue
        r,cm=run_scenario(meth,scname,classes); results.append(r); cms[(scname,meth)]=cm
        print(f"  {scname:20s} {meth:10s} F1={r['F1_macro']:.3f} Acc={r['Accuracy']:.3f} [{r['wall_s']}s]")
    log_stage(f"{scname} done")
ladder_table=pd.DataFrame(results)
print("\n=== Hasil Skema Ladder (block-split, ANN-LM=lbfgs) ===")
print(ladder_table.to_string(index=False))
ladder_table.to_csv("exports/ladder_results.csv",index=False)
print("\n[Saved] exports/ladder_results.csv"); display(ladder_table)

In [ ]:
# === Plot F1 + Accuracy per skenario (EDM vs JSD) ===
order=list(LADDER.keys())
fig,axes=plt.subplots(1,2,figsize=(15,5))
for ax,metric in zip(axes,["F1_macro","Accuracy"]):
    piv=ladder_table.pivot(index="Scenario",columns="Method",values=metric).reindex(order)
    piv.plot(kind="bar",ax=ax,ylim=(0,1.05),width=0.75)
    ax.set_title(f"Skema Ladder — {metric}"); ax.set_ylabel(metric); ax.grid(True,axis="y",alpha=0.3)
    ax.tick_params(axis="x",rotation=25)
    for c in ax.containers: ax.bar_label(c,fmt="%.2f",fontsize=8)
plt.suptitle("EDM-Fuzzy vs JSD-Fuzzy — 5 Skenario (block-split, ANN-LM/lbfgs)",fontweight="bold")
plt.tight_layout(); plt.savefig("exports/ladder_performance.png",dpi=150,bbox_inches="tight"); plt.show()
print("[Saved] exports/ladder_performance.png")

In [ ]:
# === Confusion matrix per skenario (metode terbaik) ===
for scname in LADDER:
    best_m=max(METHODS,key=lambda mm: next((r["F1_macro"] for r in results if r["Scenario"]==scname and r["Method"]==mm),0))
    if (scname,best_m) not in cms: continue
    yte,pred,names=cms[(scname,best_m)]; n=len(names)
    fig,ax=plt.subplots(figsize=(max(5,n*1.1),max(4,n)))
    cm=confusion_matrix(yte,pred,labels=np.arange(n))
    ConfusionMatrixDisplay(cm,display_labels=names).plot(ax=ax,xticks_rotation=40,colorbar=False)
    ax.set_title(f"{scname} — {best_m} (F1={next(r['F1_macro'] for r in results if r['Scenario']==scname and r['Method']==best_m)})")
    plt.tight_layout(); plt.savefig(f"exports/ladder_cm_{scname}.png",dpi=120,bbox_inches="tight"); plt.show()
print("[Saved] confusion matrices per skenario")

## Catatan penyelarasan dengan skema pembimbing

| Aspek | Skema PDF | Implementasi di sini |
|---|---|---|
| Broker | kumpulkan 4 sensor jadi 1 dataset, identitas dipertahankan | CSV gabungan = output broker; 4 channel dipertahankan |
| Entropy | EDM-Fuzzy per sensor, τ=1–10 → 40 fitur | ✓ (EDM-Fuzzy 40; + JSD-Fuzzy sebagai pembanding) |
| Skenario | ladder by multiplicity: S1 biner, S2 single(5), S3 two(7), S4 three(5), S5 four(2) | ✓ persis |
| Split | blok waktu / sumber (hindari leakage) | ✓ **block-split** 75% awal=train, 25% akhir=test per kondisi |
| Normalisasi | param dari training saja | ✓ `StandardScaler` di dalam Pipeline (fit di train fold) |
| Hidden layer | Grid Search per skenario (32..256, dua/tiga layer) | ✓ grid PDF |
| Output neuron | per skenario (2/5/7/5/2) | ✓ otomatis dari jumlah kelas |
| **ANN-LM** | Levenberg–Marquardt | ⚠️ sklearn tak punya LM → `solver='lbfgs'` (quasi-Newton, paling dekat). **LM asli → MATLAB `trainlm`** |

**Keputusan window-size.** PDF menyebut N=2.000/7.000/10.000 sampel. Kalau tiap window sebesar itu, jumlah window (dari data ~15k setelah downsample) terlalu sedikit untuk melatih ANN 5–7 kelas. Di sini dipakai **WIN=512, STRIDE=64** agar jumlah sampel cukup, dengan **block-split** menjaga train/test tidak bocor. N=2.000/7.000/10.000 lebih tepat sebagai **studi sensitivitas panjang data** (ada di notebook utama), bukan ukuran window klasifikasi. *Bisa diubah bila pembimbing memang ingin window 2.000+ (perlu data lebih panjang atau overlap lebih rapat).*